In [1]:
import pandas as pd

def separte_critical_care_list(file_path):
    df = pd.read_excel(file_path)

    # Clean up whitespace in Drug and Route
    df["Drug"] = df["Drug"].astype(str).str.strip()
    df["Route"] = df["Route"].astype(str).str.upper().str.strip()

    # Initialize route lists
    cc_drip, cc_iv, cc_im, cc_sq, cc_in = [], [], [], [], []

    # Iterate rows
    for _, row in df.iterrows():
        drug = row["Drug"]
        routes = str(row["Route"]).replace(" ", "")  # remove spaces like "IV | IM"
        route_list = routes.split("|")               # split multiple routes
        
        for r in route_list:
            if r == "DRIP":
                cc_drip.append(drug.upper().strip())
            elif r == "IV":
                cc_iv.append(drug.upper().strip())
            elif r == "IM":
                cc_im.append(drug.upper().strip())
            elif r == "SQ":
                cc_sq.append(drug.upper().strip())
            elif r == "IN":
                cc_in.append(drug.upper().strip())

    # Remove duplicates (optional)
    cc_drip = sorted(set(cc_drip))
    cc_iv   = sorted(set(cc_iv))
    cc_im   = sorted(set(cc_im))
    cc_sq   = sorted(set(cc_sq))
    cc_in   = sorted(set(cc_in))

    return cc_drip,cc_iv,cc_im,cc_sq,cc_in
def normalize_route(route):
    """Standardize route names."""
    route = str(route).upper().strip()
    if any(x in route for x in ["INFUSION", "INJECTION", "IVPB", "IV", "IM", "DRIP"]):
        return "PARENTERAL"
    elif any(x in route for x in ["SYRINGE", "SUBCUTANEOUS", "SQ"]):
        return "SQ"
    elif any(x in route for x in ["NASAL", "INTRANASAL"]):
        return "INTRANASAL"
    return "OTHER"

# --- Classification loop ---
def classify_fix_order_medication(df_fix):
    # --- Buckets for results ---
    critical_care_db = []
    critical_care_db_sq = []
    critical_care_db_in = []
    parental_control = []
    toxicity_monitoring = []
    intg_dict_hits = []
    not_found = []
    for _, row in df_fix.iterrows():
        med = str(row["Medication"]).upper().strip()
        brand = str(row["Brand"]).upper().strip()
        route = str(row["Route"]).strip()
        route_details=str(row["Route Details"]).strip()
        order_type0=str(row["Order"]).strip()
        order_type=order_type0+" MED"
         
        
        route_norm = normalize_route(row["Route"])

        med_or_brand = med if med else (brand if brand else "UNKNOWN MEDICATION")
        order_str = f"{order_type}:{med}({brand}) + {route}({route_details})"

        # Case 1: Parenteral
        if route_norm == "PARENTERAL":
            if med in cc_drip_iv_cm_u or brand in cc_drip_iv_cm_u:
                critical_care_db.append(order_str + " > Critical Care Database")
            elif med in parental_drug_db_list_u or brand in parental_drug_db_list_u:
                parental_control.append(order_str + " > Parental Control")
            elif med in toxicity_drug_db_list_u or brand in toxicity_drug_db_list_u:
                toxicity_monitoring.append(order_str + " > Toxicity Monitoring Drug")
            else:
                not_found.append(order_str + " > Not Found")

        # Case 2: SQ
        elif route_norm == "SQ":
            if med in cc_sq_u or brand in cc_sq_u:
                critical_care_db_sq.append(order_str + " > Critical Care Database SQ")
            else:
                not_found.append(order_str + " > Not Found")

        # Case 3: INTRANASAL
        elif route_norm == "INTRANASAL":
            if med in cc_in_u or brand in cc_in_u:
                critical_care_db_in.append(order_str + " > Critical Care Database INTRANASAL")
            else:
                not_found.append(order_str + " > Not Found")

        # Case 4: All others → integrated dict
        else:
            if med in intg_drug_db_dict_u:
                intg_dict_hits.append(order_str + f" > {intg_drug_db_dict_u[med]}")
            elif brand in intg_drug_db_dict_u:
                intg_dict_hits.append(order_str + f" > {intg_drug_db_dict_u[brand]}")
            else:
                not_found.append(order_str + " > Not Found")

    # # ✅ Now you have lists like:
    # print("Critical Care DB:", critical_care_db)
    # print("Critical Care DB SQ:", critical_care_db_sq)
    # print("Critical Care DB IN:", critical_care_db_in)
    # print("Parental Control:", parental_control)
    # print("Toxicity Monitoring:", toxicity_monitoring)
    # print("Integrated DB:", intg_dict_hits)
    # print("Not Found:", not_found)

    return critical_care_db,critical_care_db_sq,critical_care_db_in,parental_control,toxicity_monitoring,intg_dict_hits,not_found


# Load Excel
critical_care_path = r"./Med_Database_from_Coders/Critical_care_Updated.xlsx"
intg_drug_db =  pd.read_excel("./Med_Database_from_Coders/Integrated_Drugs_Output.xlsx")
parental_drug_db =  pd.read_excel("./Med_Database_from_Coders/Parentral control drug.xlsx")
toxicity_drug_db =  pd.read_excel("./Med_Database_from_Coders/Toxicity_drugs.xlsx")
cc_drip,cc_iv,cc_im,cc_sq,cc_in=separte_critical_care_list(critical_care_path)
cc_drip_iv_cm=cc_drip+cc_iv+cc_im

intg_drug_db_dict = dict(zip(intg_drug_db["Drug_Name"],intg_drug_db["Availability"]))
intg_drug_db_dict = {str(k).upper(): v for k, v in intg_drug_db_dict.items()}

parental_drug_db_list = parental_drug_db["Drugs"].to_list()+parental_drug_db["other name"].to_list()
parental_drug_db_list = [str(d).upper().strip() for d in parental_drug_db_list]

toxicity_drug_db_list = [d.upper().strip() for d in toxicity_drug_db["DRUGS"].to_list()]
parental_drug_db_list = list(set(parental_drug_db_list))
toxicity_drug_db_list = list(set(toxicity_drug_db_list))
parental_drug_db_list = [x for x in parental_drug_db_list if x not in ("", None) and not (isinstance(x, float) and math.isnan(x))]
toxicity_drug_db_list = [x for x in toxicity_drug_db_list if x not in ("", None) and not (isinstance(x, float) and math.isnan(x))]

# --- Precompute uppercase sets for fast lookup ---
cc_drip_iv_cm_u = set(x.upper() for x in cc_drip_iv_cm)
cc_sq_u = set(x.upper() for x in cc_sq)
cc_in_u = set(x.upper() for x in cc_in)
parental_drug_db_list_u = set(x.upper() for x in parental_drug_db_list)
toxicity_drug_db_list_u = set(x.upper() for x in toxicity_drug_db_list)
intg_drug_db_dict_u = {k.upper(): v for k, v in intg_drug_db_dict.items()}


parental_drug_db_list_u = [d for d in parental_drug_db_list_u if d != "NAN"]
toxicity_drug_db_list_u = [d for d in toxicity_drug_db_list_u if d != "NAN"]
cc_drip_iv_cm_u = [d for d in cc_drip_iv_cm_u if d != "NAN"]
cc_sq_u = [d for d in cc_sq_u if d != "NAN"]
cc_in_u = [d for d in cc_in_u if d != "NAN"]


In [32]:
import re
from mdm3_keyword import *
def extract_surgery_keywords(text, keyword_list):
    pattern = "|".join(map(re.escape, keyword_list))
    if isinstance(text, list):
        text = " ".join(text)
    found = bool(re.search(pattern, text, flags=re.IGNORECASE))
    matched = [s for s in keyword_list if re.search(re.escape(s), text, flags=re.IGNORECASE)]
    return [{found:matched}]
    #return pd.Series([found, matched], index=[f"{colunm_name}_Found", f"{colunm_name}_Surgery"])


# text="xyz Abdominal hysterectomy counseled extensively  anticoagulation risk of infection Trauma injuries financial constraints Abdominal hysterectomy  PHYSICAL THERAPY GARGLES"

def Find_MDM3_V2(chart_text,High_Risk_Medication,Moderate_Risk_Medication):
    text=chart_text
    Surgery=extract_surgery_keywords(text,Surgery_list_1)
    Surgery_discussion=extract_surgery_keywords(text,Surgery_list_1)
    Risk_factors=extract_surgery_keywords(text,Risk_factors_list_3)
    # Risk_phrases=extract_surgery_keywords(text,Risk_phrases_4)
    EMERGENCY_MAJOR_SURGERY_Decision=extract_surgery_keywords(text,EMERGENCY_MAJOR_SURGERY)
    Sdoh=extract_surgery_keywords(text,Sdoh_phrases)
    Elective_major_surgery_without_risk_discussion=extract_surgery_keywords(text,Elective_major_surgery_without_risk)
    DNR_Hospitalization_discussion=extract_surgery_keywords(text,DNR_Hospitalization)
    Therapy=extract_surgery_keywords(text,Therapy_list_low)
    Mdm_minimal_list=extract_surgery_keywords(text,Mdm_minimal)

    # print("Surgery",Surgery)
    # print("Surgery_discussion",Surgery_discussion)
    # print("Risk_factors",Risk_factors)
    # print("Risk_phrases",Risk_phrases)
    # print("EMERGENCY_MAJOR_SURGERY_Decision",EMERGENCY_MAJOR_SURGERY_Decision)
    # print("Sdoh",Sdoh)
    # print("Elective_major_surgery_without_risk_discussion",Elective_major_surgery_without_risk_discussion)
    # print("DNR_Hospitalization_discussion",DNR_Hospitalization_discussion)
    # print("Therapy",Therapy)
    # print("Mdm_minimal_list",Mdm_minimal_list)


    # ✅ Helper to extract matches safely
    def get_matches(result):
        if isinstance(result, list) and result and isinstance(result[0], dict):
            key = list(result[0].keys())[0]
            return result[0][key]
        return []
    # ✅ Build risk categories
    risk_high = []
    risk_moderate = []
    risk_low = []

    



    # High
    if get_matches(Surgery):
        risk_high.append("Decision for Surgery:"+",".join(get_matches(Surgery)))
    if get_matches(Risk_factors):
        risk_high.append("Phrase:"+",".join(get_matches(Risk_factors)))
    # if get_matches(Risk_phrases):
    #     risk_high.append("Phrases:"+",".join(get_matches(Risk_phrases)))
    if get_matches(EMERGENCY_MAJOR_SURGERY_Decision):
        risk_high.append("Decision for EMERGENCY MAJOR SURGERY:"+",".join(get_matches(EMERGENCY_MAJOR_SURGERY_Decision)))
    if get_matches(DNR_Hospitalization_discussion):
        risk_high.append("DNR/Hopitalization:"+",".join(get_matches(DNR_Hospitalization_discussion)))
    if get_matches(Elective_major_surgery_without_risk_discussion):
        risk_high.append("Elective Major Surgery:"+",".join(get_matches(Elective_major_surgery_without_risk_discussion)))

    # Moderate
    if get_matches(Therapy):
        risk_moderate.append("Therapy Suggested:"+",".join(get_matches(Therapy)))
    if get_matches(Sdoh):
        risk_moderate.append("SDOH:"+",".join(get_matches(Sdoh)))
    if get_matches(Surgery_discussion):
        risk_moderate.append("Decision for Minor Surgery:"+",".join(get_matches(Surgery_discussion)))

    # Low
    if get_matches(Mdm_minimal_list):
        risk_low.append("Minimal:"+",".join(get_matches(Mdm_minimal_list)))

    # ✅ Final result dict

    if len(High_Risk_Medication)>0:
        risk_high.extend(High_Risk_Medication)
    
    if len(Moderate_Risk_Medication)>0:
        risk_moderate.extend(Moderate_Risk_Medication)

    if len(risk_high)>0:
        MDM3_Level="HIGH"
        risk_result = {
        "High Risk": list(set(risk_high)),        # unique items
        "Moderate Risk": list(set(risk_moderate)),
        "Low Risk": list(set(risk_low))
    }
    elif len(risk_moderate)>0:
        MDM3_Level="MODERATE"
        risk_result = {        # unique items
        "Moderate Risk Factors": list(set(risk_moderate)),
        "Low Risk Factors": list(set(risk_low))
    }
    else:
        MDM3_Level="LOW"
        risk_result = {
        "Low Risk Factors": list(set(risk_low))
    }

    return MDM3_Level,risk_result

import re

def extract_acp_minutes(text: str) -> int:
    """
    Extract minutes only if the text contains
    'Time Spent on Advanced Care Planning'.
    Otherwise returns 0.
    """
    if not isinstance(text, str):
        return 0
    
    # strict regex: must contain 'Time Spent on Advanced Care Planning'
    match = re.search(r'Time Spent on Advanced Care Planning:\s*(\d+)\s*minute', 
                      text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))
    return 0


def map_acp_minutes_to_cpt(minutes: int) -> list:
    """
    Map ACP minutes to CPT codes:
      - <16 minutes → No CPT
      - 16–30 minutes → 99497
      - >30 minutes → 99497 + one or more 99498 for each additional 30 mins
    Returns a list of CPT codes.
    """
    if minutes < 16:
        return []   # No billable CPT
    
    codes = ["99497"]  # always start with 99497 if >=16 min
    if minutes > 30:
        extra = (minutes - 30 + 29) // 30   # ceiling division
        codes.extend(["99498"] * extra)
    return codes


def extract_critical_care_lines_and_max(text: str):
    """
    Extracts all lines containing 'Total critical care time'
    and returns (list_of_lines, max_minutes).
    
    - list_of_lines: all matched lines as strings
    - max_minutes: maximum integer minutes found (or 0 if none)
    """
    if not isinstance(text, str):
        return [], 0

    matched_lines = []
    minutes_list = []

    for line in text.splitlines():
        match = re.search(r'Total\s+critical\s+care\s+time\s*:\s*(\d+)\s*minute',
                          line, flags=re.IGNORECASE)
        if match:
            matched_lines.append(line.strip())
            minutes_list.append(int(match.group(1)))

    max_minutes = max(minutes_list) if minutes_list else 0
    return matched_lines, max_minutes


def map_critical_care_minutes_to_cpt(minutes: int) -> list:
    """
    Maps critical care time in minutes to CPT codes.
    
    - <30 minutes → []
    - 30–74 minutes → [99291]
    - ≥75 minutes → [99291] + one or more 99292
    """
    if minutes < 30:
        return []   # Not billable
    
    codes = ["99291"]
    if minutes > 74:
        extra = (minutes - 74 + 29) // 30   # ceiling division for additional 30-min blocks
        codes.extend(["99292"] * extra)
    return codes



In [33]:
import os
import pandas as pd
import glob

file_path = r"./All_Headers_Data\BATCH_1_CSCC_All_Header_Data_From_Mango_DB.xlsx"
Medication_Data = r"./Found_Med_from_order_and_chart_BATCH1"

uncategorized_medication=[]

# Read the first sheet (default)
df = pd.read_excel(file_path)

df["DOS"] = pd.to_datetime(df["DOS"], errors="coerce").dt.date
combine_groups = {
    'data_for_medication': ['Assessment', 'Assessment & Plan','Assessment/Plan','Code Status','Current Facility-Administered Medications','Current Outpatient Medications', 'Discharge Diagnosis','Hospital Course','MEDICATIONS'],
}


columns_to_keep = ['Sr No', 'Filename', 'DOS', 'Visit']

for new_col, cols_to_combine in combine_groups.items():
    df[new_col] = ""
    for col in cols_to_combine:
        if col in df.columns:  # Safe: skip missing columns
            df[new_col] = df[new_col] + " " + df[col].astype(str)
    df[new_col] = df[new_col].str.strip()

df_final = df[columns_to_keep + list(combine_groups.keys())]

df_final["DOS"] = pd.to_datetime(df_final["DOS"], errors="coerce").dt.date

unique_filenames = df_final["Filename"].unique()
selected_filenames = unique_filenames[:11]  # pick first 11
selected_filenames = unique_filenames
results = []
i = 0
for main_filename in selected_filenames:
    # print(i)
    i += 1
    # print("Filename:", main_filename)
    main_filename1 = main_filename.replace(".TIF", "").strip()

    # get all DOS for this filename
    dos_visit_list = df_final.loc[df_final["Filename"] == main_filename, ["DOS", "Visit"]].drop_duplicates().values.tolist()
  

    for date, visit_type in dos_visit_list:
        sub_df = df[
                (df["Filename"] == main_filename) &
                (df["DOS"] == date) &
                (df["Visit"] == visit_type)
            ]
        print("Filename:", main_filename)
        print("DOS:", date, "Visit:", visit_type)
        critical_care_text = "\n".join(sub_df["critical care time"].dropna().astype(str))
    
        # print("Critical Care Text:\n",critical_care_text)
        cc_lines,cc_time=extract_critical_care_lines_and_max(critical_care_text)
        print("\n".join(cc_lines))
        print(cc_time)
        cc_codes=map_critical_care_minutes_to_cpt(cc_time)
        print(",".join(cc_codes))
        print("-"*40)
#         # print("Filename:", main_filename)
#         # print("DOS:", date, "Visit:", visit_type)

#         if visit_type=="acp":
#             print("Filename:", main_filename)
#             print("DOS:", date, "Visit:", visit_type)
#             # print("YEs")

#             # Filter rows for this filename, DOS, Visit
#             sub_df = df[
#                 (df["Filename"] == main_filename) &
#                 (df["DOS"] == date) &
#                 (df["Visit"] == visit_type)
#             ]

#             # Combine all column values into one big string
#             if not sub_df.empty:
#                 # Collapse each row into a single string, then join them with newline
#                 combined_text = "\n".join(sub_df.astype(str).apply(lambda row: " | ".join(row), axis=1))

#                 acp_time=extract_acp_minutes(combined_text)
#                 acp_cpt=map_acp_minutes_to_cpt(acp_time)
#                 if acp_time==0:
#                     risk_str="Time Spent on Advanced Care Planning: Missing\n"+"ACP CPT: None"
#                 else:
#                     risk_str="Time Spent on Advanced Care Planning(Minutes):"+ str(acp_time) + " minutes\n"+"ACP CPT:"+",".join(acp_cpt)
#                 print("Time Spent on Advanced Care Planning(Minutes):",acp_time)                
#                 print("ACP CPT Code:",acp_cpt)

#             else:
#                 print("⚠️ No rows found for", main_filename, date, visit_type)
#                 risk_str="Missing Time Spent on Advanced Care Planning "
#                 acp_cpt=[]

#             # print("-"*40)

#             chart_data=chart_data.replace(";",", ")
#             chart_data=chart_data.replace("\n","~ ")
#             chart_name=main_filename.replace("CSCC_","").replace(".TIF","").strip()
#             row_info = {
#                 # optional, or keep from df_final if needed
#                 "Filename": main_filename,
#                 "Chart":chart_name,
#                 "DOS": date,
#                 "Visit": visit_type.strip(),
#                 "MDM3_Level": ",".join(acp_cpt),
#                 "Extracted_Data_for_MDM3": risk_str
#             }
#             results.append(row_info)
#         else:

#             output_filename = f"{main_filename1}_{date}_{visit_type}_MED.xlsx"
#             Medication_Extrated_Folder = os.path.join(Medication_Data, output_filename)

#             High_Risk_Medication = []
#             Moderate_Risk_Medication = []
#             # ✅ If file not found, search for combined filenames
#             if not os.path.exists(Medication_Extrated_Folder):
#                 search_pattern = f"{main_filename1}_{date}_*{visit_type}*_MED.xlsx"
#                 possible_files = glob.glob(os.path.join(Medication_Data, search_pattern))

#                 if possible_files:
#                     output_filename_combine = os.path.basename(possible_files[0])
#                     # print(f"Found combined file: {output_filename_combine}")
#                     Medication_Extrated_Folder = possible_files[0]

#             if os.path.exists(Medication_Extrated_Folder):
#                 try:
#                     df_out = pd.read_excel(Medication_Extrated_Folder)
#                     # print(f"✅ Opened: {Medication_Extrated_Folder}")
#                     # display(df_out)  # show first few rows
#                     critical_care_db,critical_care_db_sq,critical_care_db_in,parental_control,toxicity_monitoring,intg_dict_hits,not_found=classify_fix_order_medication(df_out)
#                     uncategorized_medication.extend(not_found)
#                     High_Risk_Medication=critical_care_db+critical_care_db_sq+critical_care_db_in+parental_control+toxicity_monitoring
#                     Moderate_Risk_Medication=intg_dict_hits

#                     print("High_Risk_Medication:",High_Risk_Medication)
#                     print("Moderate_Risk_Medication:>",Moderate_Risk_Medication)              
#                 except Exception as e:
#                     print(f"⚠️ Error reading {Medication_Extrated_Folder}: {e}")
#             else:
#                 print(f"❌ File not found: {Medication_Extrated_Folder}") 


#             # chart data (from df_final)
#             chart_data_list = df_final.loc[
#                 (df_final["Filename"] == main_filename) & (df_final["DOS"] == date),
#                 "data_for_medication"
#             ].tolist()

#             # visit_type_1 = df_final.loc[
#             #     (df_final["Filename"] == main_filename) & (df_final["DOS"] == date),
#             #     "Visit"
#             # ].tolist()
#             # visit_type_2=" ".join(visit_type_1)
#             # visit_type=visit_type_2.strip()

#             chart_data="\n".join(chart_data_list)
#             # print(chart_data)

#             MDM3_Level,risk_result=Find_MDM3_V2(chart_data,High_Risk_Medication,Moderate_Risk_Medication)
#             # risk_str = str(risk_result).replace("{","").replace("}","")
#             output_lines = []
#             for key, values in risk_result.items():
#                 if values:  # only if not empty
#                     output_lines.append(f"{key}:{values}")
#                     # output_lines.extend(values)

#             risk_str = "~".join(output_lines)
#             # print(final_output)




#             print("MDM3_Level",MDM3_Level)
#             print("risk_result",risk_result)
#             # ✅ Save into results list
            
#             chart_data=chart_data.replace(";",", ")
#             chart_data=chart_data.replace("\n","~ ")
#             chart_name=main_filename.replace("AHFL_","").replace(".TIF","").strip()
#             row_info = {
#                 # optional, or keep from df_final if needed
#                 "Filename": main_filename,
#                 "Chart":chart_name,
#                 "DOS": date,
#                 "Visit": visit_type.strip(),
#                 "MDM3_Level": MDM3_Level,
#                 "Extracted_Data_for_MDM3": risk_str
#             }
#             results.append(row_info)

#         print("-"*40)

# # ✅ Convert results to DataFrame
# df_results = pd.DataFrame(results)
# # --- Format DOS column as MM/DD/YYYY ---
# # df_results["DOS"] = pd.to_datetime(df_results["DOS"], errors="coerce").dt.strftime("%m/%d/%Y")
# df_results["DOS"] = pd.to_datetime(df_results["DOS"], errors="coerce")
# df_results["DOS"] = (
#     df_results["DOS"].dt.month.astype(str) + "/" +
#     df_results["DOS"].dt.day.astype(str) + "/" +
#     df_results["DOS"].dt.year.astype(str)
# )

# # Save final output Excel
# output_file = "./OUTPUT/BATCH1_CSCC_MDM3_Results_4_10_2025.xlsx"
# df_results.to_excel(output_file, index=False)
# # print(f"\n✅ Saved results to {output_file}")


# output_csv = r"./OUTPUT/BATCH1_CSCC_MDM3_Results_4_10_2025.csv"
# df_results.to_csv(output_csv, index=False, sep=";")

# print(f"✅ Saved CSV:   {output_csv}")

        

        

C:\Users\pbhopale\AppData\Local\Temp\ipykernel_19644\237977891.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final["DOS"] = pd.to_datetime(df_final["DOS"], errors="coerce").dt.date


Filename: CSCC_HMV001969162.TIF
DOS: 2024-10-02 Visit: subsequent
Total critical care time: 38 minutes
Total critical care time: 37 minutes
38
99291
----------------------------------------
Filename: CSCC_HMV001969162.TIF
DOS: 2024-10-03 Visit: subsequent
Total critical care time: 31 minutes
31
99291
----------------------------------------
Filename: CSCC_HMV001969162.TIF
DOS: 2024-10-02 Visit: acp

0

----------------------------------------
Filename: CSCC_HMV001969353.TIF
DOS: 2024-10-02 Visit: initial
Total critical care time:45 minutes
45
99291
----------------------------------------
Filename: CSCC_HMV001969353.TIF
DOS: 2024-10-02 Visit: subsequent
Total critical care time:40 minutes
40
99291
----------------------------------------
Filename: CSCC_HMV001969353.TIF
DOS: 2024-10-03 Visit: subsequent
Total critical care time: 42 minutes
42
99291
----------------------------------------
Filename: CSCC_HMV001969353.TIF
DOS: 2024-10-04 Visit: subsequent
Total critical care time: 40 minu

In [8]:
import re
uncategorized_medication2=list(set(uncategorized_medication))


rows = []
for item in uncategorized_medication:
    # Split into left and right
    left, right = item.split(" + ", 1)
    route_full, not_found = right.split(" > ")
    
    # Extract FIX/PRN
    fix_prn, rest = left.split(":", 1)
    
    # Extract Medication and Brand
    med_match = re.match(r"([A-Z0-9 %]+)(?:\(([^)]+)\))?", rest.strip())
    if med_match:
        medication = med_match.group(1).strip()
        brand = med_match.group(2).strip() if med_match.group(2) else ""
    else:
        medication, brand = rest.strip(), ""
    
    # Get only route initial before "("
    route_initial = route_full.split("(")[0].strip()
    
    rows.append([fix_prn, medication, brand, route_initial, not_found.strip()])

# Create DataFrame
df = pd.DataFrame(rows, columns=["FIX/PRN", "Medication", "Brand", "Route", "Not Found"])

# Drop duplicates
df = df.drop_duplicates()

# Add serial numbers
df.insert(0, "Sr.", range(1, len(df) + 1))

# Save to Excel
df.to_excel("Uncategorized_Medications_Clean.xlsx", index=False)